# Polynomial-zonotope vs interval adaptive quadrature

This notebook compares certified adaptive quadrature enclosures from the existing interval machinery with the polynomial-zonotope (PZ) two-jet machinery. It uses a small tanh network and a fixed box domain in the same style as the interval norm and PINN notebooks: `IntervalTensor` domains, monkey-patched PyTorch modules, and adaptive `model.lpnorm(...)` / `model.sobolev_norm(...)` calls.

The defaults are intentionally small and reproducible so the notebook can be run quickly in CI-like or laptop environments.

## 1) Setup and reproducibility

In [ ]:
from __future__ import annotations

import math
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from intervalnets import Interval, IntervalTensor, PolynomialZonotope, enable_interval_eval
from intervalnets.pz_integration import _split_box as _pz_split_box, _choose_split_dim_from_jacobian, _dorfler_marking as _pz_dorfler_marking, _integrated_squared_contribution, _interval_add, _interval_width, _sqrt_interval_nonnegative
from intervalnets.pytorch import _box_volume, _choose_split_dim, _dorfler_marking, _interval_pow_scalar, _jacobian_is_exact_zero, _hessian_is_exact_zero, _interval_tensor_is_exact_constant, _lp_pointwise_power_bounds_refined, _sobolev_pointwise_power_bounds_refined, _split_box

enable_interval_eval()
SEED = 20260720
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
print(f"repo_root={repo_root}")
print(f"torch={torch.__version__}, seed={SEED}")

## 2) Small model and domain

The network is deliberately tiny, with `Tanh` activations so second derivatives are meaningful. The domain is a flat `IntervalTensor` box, matching the current adaptive norm APIs.

In [ ]:
def make_small_tanh_network(input_dim: int = 2, width: int = 6, hidden_layers: int = 2) -> nn.Sequential:
    layers: list[nn.Module] = []
    in_features = input_dim
    for _ in range(hidden_layers):
        layers += [nn.Linear(in_features, width), nn.Tanh()]
        in_features = width
    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers).to(dtype=torch.float64)

model = make_small_tanh_network()
with torch.no_grad():
    for i, param in enumerate(model.parameters()):
        torch.manual_seed(SEED + i)
        param.copy_(0.35 * torch.randn_like(param))

domain = IntervalTensor.from_bounds(torch.tensor([-1.0, -0.75]), torch.tensor([1.0, 0.75]))
MAX_ITERATIONS = 4
ITERATIONS = list(range(MAX_ITERATIONS + 1))
THETA = 0.5
CHEBYSHEV_DEGREE = 3
RESIDUAL_SUBDIVISIONS = 32
FORWARD_REFINE_SPLITS = 1
FORWARD_REFINE_MAX_CELLS = 256
model, domain, ITERATIONS

## 3) Helpers for timing, cell counts, and diagnostics

In [ ]:
def interval_width(bounds) -> float:
    return float(bounds.upper) - float(bounds.lower)

def time_call(fn):
    t0 = time.perf_counter()
    value = fn()
    return value, time.perf_counter() - t0

def interval_quantity_power_bounds(model, box, quantity: str):
    if quantity == "L2":
        return _lp_pointwise_power_bounds_refined(
            model,
            box,
            2.0,
            forward_refine_splits=FORWARD_REFINE_SPLITS,
            forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS,
        )
    if quantity == "W12":
        return _sobolev_pointwise_power_bounds_refined(
            model,
            box,
            2.0,
            order=1,
            forward_refine_splits=FORWARD_REFINE_SPLITS,
            forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS,
        )
    if quantity == "W22":
        return _sobolev_pointwise_power_bounds_refined(
            model,
            box,
            2.0,
            order=2,
            forward_refine_splits=FORWARD_REFINE_SPLITS,
            forward_refine_max_cells=FORWARD_REFINE_MAX_CELLS,
        )
    raise ValueError(quantity)

def interval_cell_indicator_and_split_dim(model, box, quantity: str) -> tuple[float, int]:
    integrand_bounds = interval_quantity_power_bounds(model, box, quantity)
    width = interval_width(integrand_bounds)
    if quantity == "L2":
        jacobian = model.eval_jacobian(box) if len(box.lower) > 1 else None
        return width * _box_volume(box), _choose_split_dim(box, jacobian)

    order = 1 if quantity == "W12" else 2
    output = model.eval(box)
    jacobian = model.eval_jacobian(box)
    hessian = model.eval_hessian(box) if order == 2 else None
    derivative_zero = _jacobian_is_exact_zero(jacobian)
    second_derivative_zero = True if hessian is None else _hessian_is_exact_zero(hessian)
    indicator = 0.0 if (_interval_tensor_is_exact_constant(output) and derivative_zero and second_derivative_zero) else width * _box_volume(box)
    return indicator, _choose_split_dim(box, jacobian if len(box.lower) > 1 else None)

def interval_aggregate_bounds(model, boxes, quantity: str):
    integral = Interval.point(0.0)
    for box in boxes:
        integrand_bounds = interval_quantity_power_bounds(model, box, quantity)
        weighted = Interval.from_bounds(
            float(integrand_bounds.lower) * _box_volume(box),
            float(integrand_bounds.upper) * _box_volume(box),
        )
        integral = integral + weighted
    non_negative = Interval.from_bounds(max(0.0, float(integral.lower)), max(0.0, float(integral.upper)))
    return _interval_pow_scalar(non_negative, 0.5)

def pz_complexity(model, domain) -> dict[str, int]:
    pz_domain = PolynomialZonotope.from_box(domain.lower, domain.upper)
    jet = model.eval_pz_twojet(pz_domain, chebyshev_degree=CHEBYSHEV_DEGREE, residual_subdivisions=RESIDUAL_SUBDIVISIONS)
    polynomials = [pz for pz in [jet.Y, jet.J, jet.H] if pz is not None]
    max_num_noise = max((pz.num_noise for pz in polynomials), default=pz_domain.num_noise)
    return {
        "pz_terms": int(sum(len(pz.terms) for pz in polynomials)),
        "pz_noise_vars": int(max_num_noise),
        "pz_approx_noise_vars": int(max_num_noise - pz_domain.num_noise),
    }


## 4) Certified interval and PZ quadrature comparisons

In [ ]:
def make_trace_row(quantity: str, method: str, iteration: int, bounds, cells: int, seconds: float, meta: dict[str, int]):
    return {
        "quantity": quantity,
        "method": method,
        "iteration": iteration,
        "lower": float(bounds.lower),
        "upper": float(bounds.upper),
        "width": interval_width(bounds),
        "cells": cells,
        "seconds": seconds,
        **meta,
    }

def run_interval_adaptive_trace(model, domain, quantity: str, max_iterations: int):
    boxes = [domain]
    rows = []
    elapsed = 0.0
    meta = {"pz_terms": 0, "pz_noise_vars": 0, "pz_approx_noise_vars": 0}
    for iteration in range(max_iterations + 1):
        t0 = time.perf_counter()
        bounds = interval_aggregate_bounds(model, boxes, quantity)
        elapsed += time.perf_counter() - t0
        rows.append(make_trace_row(quantity, "interval", iteration, bounds, len(boxes), elapsed, meta))
        if iteration == max_iterations:
            break

        t0 = time.perf_counter()
        indicators, split_dims = zip(*(interval_cell_indicator_and_split_dim(model, box, quantity) for box in boxes))
        marked = set(_dorfler_marking(list(indicators), THETA))
        boxes = [child for idx, box in enumerate(boxes) for child in (_split_box(box, split_dim=split_dims[idx]) if idx in marked else (box,))]
        elapsed += time.perf_counter() - t0
    return rows

def pz_kind(quantity: str) -> str:
    return {"L2": "l2", "W12": "w12", "W22": "w22"}[quantity]

def pz_aggregate_bounds(model, boxes, quantity: str):
    integral = Interval.point(0.0)
    for box in boxes:
        contribution, _ = _integrated_squared_contribution(
            model,
            box,
            integrand_kind=pz_kind(quantity),
            chebyshev_degree=CHEBYSHEV_DEGREE,
            residual_subdivisions=RESIDUAL_SUBDIVISIONS,
        )
        integral = _interval_add(integral, contribution)
    return _sqrt_interval_nonnegative(integral)

def run_pz_adaptive_trace(model, domain, quantity: str, max_iterations: int):
    boxes = [domain]
    rows = []
    elapsed = 0.0
    meta = pz_complexity(model, domain)
    for iteration in range(max_iterations + 1):
        t0 = time.perf_counter()
        bounds = pz_aggregate_bounds(model, boxes, quantity)
        elapsed += time.perf_counter() - t0
        rows.append(make_trace_row(quantity, "pz", iteration, bounds, len(boxes), elapsed, meta))
        if iteration == max_iterations:
            break

        t0 = time.perf_counter()
        contributions_and_jacobians = [
            _integrated_squared_contribution(
                model,
                box,
                integrand_kind=pz_kind(quantity),
                chebyshev_degree=CHEBYSHEV_DEGREE,
                residual_subdivisions=RESIDUAL_SUBDIVISIONS,
            )
            for box in boxes
        ]
        indicators = [_interval_width(contribution) for contribution, _ in contributions_and_jacobians]
        split_dims = [_choose_split_dim_from_jacobian(box, jacobian) for box, (_, jacobian) in zip(boxes, contributions_and_jacobians)]
        marked = set(_pz_dorfler_marking(indicators, THETA))
        boxes = [child for idx, box in enumerate(boxes) for child in (_pz_split_box(box, split_dim=split_dims[idx]) if idx in marked else (box,))]
        elapsed += time.perf_counter() - t0
    return rows

rows = []
for quantity in ["L2", "W12", "W22"]:
    rows.extend(run_interval_adaptive_trace(model, domain, quantity, MAX_ITERATIONS))
    rows.extend(run_pz_adaptive_trace(model, domain, quantity, MAX_ITERATIONS))
results = pd.DataFrame(rows)
results


## 5) Non-certified Monte Carlo/autograd sanity check

The estimates below are not certificates. They simply check that certified lower/upper ranges are plausible for random samples and PyTorch autograd derivatives.

In [ ]:
def autograd_quantity_values(samples: torch.Tensor) -> dict[str, np.ndarray]:
    samples = samples.clone().detach().requires_grad_(True)
    y = model(samples)[:, 0]
    grad = torch.autograd.grad(y.sum(), samples, create_graph=True)[0]
    hess_sq = torch.zeros_like(y)
    for i in range(samples.shape[1]):
        for j in range(samples.shape[1]):
            hij = torch.autograd.grad(grad[:, i].sum(), samples, retain_graph=True)[0][:, j]
            hess_sq = hess_sq + hij.square()
    grad_sq = grad.square().sum(dim=1)
    return {"L2": y.square().detach().numpy(), "W12": (y.square() + grad_sq).detach().numpy(), "W22": (y.square() + grad_sq + hess_sq).detach().numpy()}

N_MC = 4096
rng = torch.Generator().manual_seed(SEED)
lo, hi = domain.lower.to(dtype=torch.float64), domain.upper.to(dtype=torch.float64)
samples = lo + (hi - lo) * torch.rand((N_MC, len(lo)), generator=rng, dtype=torch.float64)
volume = float(torch.prod(hi - lo))
values = autograd_quantity_values(samples)
mc_rows = []
for quantity, pointwise in values.items():
    estimate = math.sqrt(max(0.0, volume * float(np.mean(pointwise))))
    final = results[(results.quantity == quantity) & (results.iteration == MAX_ITERATIONS)]
    for method in ["interval", "pz"]:
        certified = final[final.method == method].iloc[0]
        mc_rows.append({"quantity": quantity, "method": method, "mc_estimate": estimate, "certified_lower": certified.lower, "certified_upper": certified.upper, "inside_certified_bounds": certified.lower <= estimate <= certified.upper})
pd.DataFrame(mc_rows)

## 6) Width-vs-refinement plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (quantity, method), group in results.groupby(["quantity", "method"]):
    ax.plot(group["iteration"], group["width"], marker="o", label=f"{quantity} / {method}")
ax.set_xlabel("refinement iteration")
ax.set_ylabel("certified bound width")
ax.set_yscale("log")
ax.set_title("Interval width versus adaptive refinement")
ax.grid(True, which="both", alpha=0.3)
ax.legend(ncol=2)
plt.tight_layout()

## 7) Compact final table

In [ ]:
final_table = results[results["iteration"] == MAX_ITERATIONS].copy()
final_table[["quantity", "method", "lower", "upper", "width", "cells", "seconds", "pz_terms", "pz_noise_vars", "pz_approx_noise_vars"]].sort_values(["quantity", "method"])